In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(
    "Data/final_customer_support.csv"
)

df.shape

(200000, 36)

In [3]:
df["category"].value_counts()

category
Feature Request              20169
Subscription Cancellation    20096
Performance Issue            20074
Security Concern             20040
Login Issue                  20002
Payment Problem              19997
Bug Report                   19981
Refund Request               19900
Data Sync Issue              19877
Account Suspension           19864
Name: count, dtype: int64

In [4]:
df["category"].nunique()

10

In [5]:
df["combined_text"] = (
    df["issue_description"].astype(str)
    + " "
    + df["resolution_notes"].astype(str)
)

df["combined_text"].head()

0    The payment was deducted from my bank account ...
1    I found a bug in the latest update affecting r...
2    The application crashes whenever I try to uplo...
3    My subscription was cancelled without my reque...
4    The system is not syncing data across devices ...
Name: combined_text, dtype: str

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["category_encoded"] = le.fit_transform(
    df["category"]
)

print(le.classes_)

['Account Suspension' 'Bug Report' 'Data Sync Issue' 'Feature Request'
 'Login Issue' 'Payment Problem' 'Performance Issue' 'Refund Request'
 'Security Concern' 'Subscription Cancellation']


In [7]:
X = df["combined_text"]

y = df["category_encoded"]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [9]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(140000,)
(30000,)
(30000,)


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)

(140000, 99)


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(
    X_train_tfidf,
    y_train
)

y_pred_lr = lr_model.predict(
    X_test_tfidf
)

lr_acc = accuracy_score(
    y_test,
    y_pred_lr
)

print(
    "Logistic Regression Accuracy:",
    round(lr_acc*100,2),
    "%"
)

Logistic Regression Accuracy: 10.04 %


In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_tfidf,
    y_train
)

y_pred_rf = rf_model.predict(
    X_test_tfidf
)

rf_acc = accuracy_score(
    y_test,
    y_pred_rf
)

print(
    "Random Forest Accuracy:",
    round(rf_acc*100,2),
    "%"
)

Random Forest Accuracy: 9.82 %


In [13]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy":[
        lr_acc,
        rf_acc
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.100433
1,Random Forest,0.098167


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.utils import to_categorical

In [15]:
max_words = 5000

tokenizer = Tokenizer(
    num_words=max_words,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [16]:
X_train_seq = tokenizer.texts_to_sequences(
    X_train
)

X_val_seq = tokenizer.texts_to_sequences(
    X_val
)

X_test_seq = tokenizer.texts_to_sequences(
    X_test
)

In [17]:
max_len = 100

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=max_len,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post"
)

print(X_train_pad.shape)

(140000, 100)


In [18]:
y_train_cat = to_categorical(y_train)

y_val_cat = to_categorical(y_val)

y_test_cat = to_categorical(y_test)

print(y_train_cat.shape)

(140000, 10)


In [19]:
num_classes = len(le.classes_)

model = Sequential()

model.add(
    Embedding(
        input_dim=max_words,
        output_dim=64,
        input_length=max_len
    )
)

model.add(
    LSTM(64)
)

model.add(
    Dropout(0.3)
)

model.add(
    Dense(
        32,
        activation="relu"
    )
)

model.add(
    Dense(
        num_classes,
        activation="softmax"
    )
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [21]:
history = model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(
        X_val_pad,
        y_val_cat
    ),
    epochs=3,
    batch_size=128
)

Epoch 1/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 168s 148ms/step - accuracy: 0.0992 - loss: 2.3028 - val_accuracy: 0.1000 - val_loss: 2.3027
Epoch 2/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 143s 131ms/step - accuracy: 0.0997 - loss: 2.3027 - val_accuracy: 0.1002 - val_loss: 2.3029
Epoch 3/3
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 453s 415ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1002 - val_loss: 2.3026


In [22]:
loss, acc = model.evaluate(
    X_test_pad,
    y_test_cat
)

print(
    "LSTM Accuracy:",
    round(acc*100,2),
    "%"
)

938/938 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.1002 - loss: 2.3026
LSTM Accuracy: 10.02 %


In [23]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "LSTM"
    ],
    "Accuracy":[
        lr_acc,
        rf_acc,
        acc
    ]
})

results

,Model,Accuracy
0,Logistic Regression,0.100433
1,Random Forest,0.098167
2,LSTM,0.100200


In [24]:
model.save(
    "models/priority_category_lstm.keras"
)

print("Model Saved Successfully")

Model Saved Successfully


In [26]:
import pickle

with open(
    "models/category_tokenizer.pkl",
    "wb"
) as f:
    pickle.dump(tokenizer, f)

with open(
    "models/category_encoder.pkl",
    "wb"
) as f:
    pickle.dump(le, f)

print("Category tokenizer and encoder saved")

Category tokenizer and encoder saved
